In [14]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import resnet101
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
import pandas as pd
import os
import cv2
import numpy as np
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

# Define the ORIGA Dataset class
class ORIGADataset(Dataset):
    def __init__(self, img_paths, cdr_values, transform=None):
        self.img_paths = img_paths
        self.cdr_values = cdr_values
        self.transform = transform
        
    def __len__(self):
        return len(self.img_paths)
    
    def __getitem__(self, idx):
        try:
            # Load image
            img_path = self.img_paths[idx]
            
            # Try different loading methods
            if os.path.exists(img_path):
                img = cv2.imread(img_path)
                if img is not None:
                    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                else:
                    # Fallback to PIL
                    img = Image.open(img_path).convert('RGB')
                    img = np.array(img)
            else:
                raise FileNotFoundError(f"Image not found: {img_path}")
            
            # Get CDR and severity
            cdr = self.cdr_values[idx]
            severity = self.cdr_to_severity(cdr)
            
            # Apply transforms
            if self.transform:
                img = self.transform(img)
                
            return img, torch.tensor(cdr, dtype=torch.float32), torch.tensor(severity, dtype=torch.long)
        
        except Exception as e:
            print(f"Error loading image {img_path}: {e}")
            # Return a dummy image in case of error
            dummy_img = np.zeros((256, 256, 3), dtype=np.uint8)
            if self.transform:
                dummy_img = self.transform(dummy_img)
            return dummy_img, torch.tensor(0.5, dtype=torch.float32), torch.tensor(0, dtype=torch.long)
    
    def cdr_to_severity(self, cdr):
        """Convert CDR to severity classification"""
        if cdr < 0.7:
            return 0  # Mild
        elif cdr < 0.9:
            return 1  # Moderate
        else:
            return 2  # Severe

# Define the Squeeze-and-Excitation (SE) Block
class SEBlock(nn.Module):
    def __init__(self, channel, reduction=16):
        super(SEBlock, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channel, channel // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channel // reduction, channel, bias=False),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)

# Define the Glaucoma Severity Model with SE Blocks
class GlaucomaSeverityModel(nn.Module):
    def __init__(self, num_classes=3):
        super(GlaucomaSeverityModel, self).__init__()
        # Load ResNet101 with updated weights parameter
        self.backbone = resnet101(weights='IMAGENET1K_V1')
        
        # Remove the original FC layer
        self.backbone.fc = nn.Identity()
        
        # Add SE Blocks to the backbone layers
        self.backbone.layer1 = nn.Sequential(self.backbone.layer1, SEBlock(256))
        self.backbone.layer2 = nn.Sequential(self.backbone.layer2, SEBlock(512))
        self.backbone.layer3 = nn.Sequential(self.backbone.layer3, SEBlock(1024))
        self.backbone.layer4 = nn.Sequential(self.backbone.layer4, SEBlock(2048))
        
        # Regression head for CDR prediction
        self.regressor = nn.Sequential(
            nn.Linear(2048, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, 1)
        )
        
        # Classification head for severity
        self.classifier = nn.Sequential(
            nn.Linear(2048, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )
        
    def forward(self, x):
        features = self.backbone(x)
        cdr = self.regressor(features)
        severity = self.classifier(features)
        return cdr, severity

# Define Focal Loss for handling class imbalance
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = nn.CrossEntropyLoss(reduction='none')(inputs, targets)
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
        
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

# Define training transforms with advanced augmentation
train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((256, 256)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(degrees=15),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=3)], p=0.3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def load_data(data_path):
    """Load and prepare the ORIGA dataset"""
    # Load the metadata
    metadata_path = os.path.join(data_path, "OrigaList.csv")
    origa_metadata = pd.read_csv(metadata_path)
    
    # Define the base directory for images
    base_dir = os.path.join(data_path, "Images_Cropped")
    
    # Print dataset info
    print(f"Loading data from: {data_path}")
    print(f"Metadata file: {metadata_path}")
    print(f"Images directory: {base_dir}")
    print(f"Metadata shape: {origa_metadata.shape}")
    print(f"Columns: {list(origa_metadata.columns)}")
    
    # Construct full image paths and filter existing files
    img_paths = []
    cdr_values = []
    
    for idx, row in origa_metadata.iterrows():
        img_path = os.path.join(base_dir, row["Filename"])
        if os.path.exists(img_path):
            img_paths.append(img_path)
            cdr_values.append(row["ExpCDR"])
        else:
            print(f"Warning: Image not found: {img_path}")
    
    print(f"Found {len(img_paths)} valid images out of {len(origa_metadata)} total entries")
    
    # Print CDR distribution
    cdr_array = np.array(cdr_values)
    print(f"CDR range: {cdr_array.min():.3f} - {cdr_array.max():.3f}")
    print(f"CDR mean: {cdr_array.mean():.3f} ± {cdr_array.std():.3f}")
    
    # Print severity distribution
    severity_counts = [0, 0, 0]  # Mild, Moderate, Severe
    for cdr in cdr_values:
        if cdr < 0.7:
            severity_counts[0] += 1
        elif cdr < 0.9:
            severity_counts[1] += 1
        else:
            severity_counts[2] += 1
    
    print(f"Severity distribution - Mild: {severity_counts[0]}, Moderate: {severity_counts[1]}, Severe: {severity_counts[2]}")
    
    return img_paths, np.array(cdr_values)

def create_data_loaders(img_paths, cdr_values, batch_size=32, val_split=0.2):
    """Create training and validation data loaders"""
    # Split the data
    train_img_paths, val_img_paths, train_cdr_values, val_cdr_values = train_test_split(
        img_paths, cdr_values, test_size=val_split, random_state=42, stratify=None
    )
    
    # Create datasets
    train_dataset = ORIGADataset(train_img_paths, train_cdr_values, transform=train_transform)
    val_dataset = ORIGADataset(val_img_paths, val_cdr_values, transform=val_transform)
    
    # Create data loaders with reduced num_workers for CPU
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=False)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=False)
    
    return train_loader, val_loader, len(train_dataset), len(val_dataset)

def train_model(model, train_loader, val_loader, train_size, val_size, epochs=20, device='cuda'):
    """Train the glaucoma severity model"""
    
    # Loss functions
    regression_loss = nn.MSELoss()
    classification_loss = FocalLoss(alpha=1, gamma=2)
    
    # Optimizer with weight decay
    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
    
    # Learning rate scheduler - removed verbose parameter
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3
    )
    
    # Training variables
    best_val_loss = float('inf')
    patience = 7
    early_stopping_counter = 0
    
    for epoch in range(epochs):
        # Training phase
        model.train()
        train_loss = 0.0
        
        print(f"Epoch {epoch+1}/{epochs} - Training...")
        
        for batch_idx, (imgs, cdr_true, severity_true) in enumerate(train_loader):
            imgs = imgs.to(device, non_blocking=True)
            cdr_true = cdr_true.to(device, non_blocking=True)
            severity_true = severity_true.to(device, non_blocking=True)
            
            optimizer.zero_grad()
            cdr_pred, severity_pred = model(imgs)
            
            # Combined loss with weights
            reg_loss = regression_loss(cdr_pred.squeeze(), cdr_true)
            cls_loss = classification_loss(severity_pred, severity_true)
            loss = 0.5 * reg_loss + 0.5 * cls_loss
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            train_loss += loss.item()
            
            # Print progress every 10 batches
            if (batch_idx + 1) % 10 == 0:
                print(f"  Batch {batch_idx+1}/{len(train_loader)}, Loss: {loss.item():.4f}")
        
        # Validation phase
        print(f"Epoch {epoch+1}/{epochs} - Validating...")
        model.eval()
        val_loss = 0.0
        cdr_mae = 0.0
        correct_predictions = 0
        all_severity_true = []
        all_severity_pred = []
        
        with torch.no_grad():
            for imgs, cdr_true, severity_true in val_loader:
                imgs = imgs.to(device, non_blocking=True)
                cdr_true = cdr_true.to(device, non_blocking=True)
                severity_true = severity_true.to(device, non_blocking=True)
                
                cdr_pred, severity_pred = model(imgs)
                
                # Calculate losses
                reg_loss = regression_loss(cdr_pred.squeeze(), cdr_true)
                cls_loss = classification_loss(severity_pred, severity_true)
                val_loss += 0.5 * reg_loss + 0.5 * cls_loss
                
                # Calculate metrics
                cdr_mae += torch.abs(cdr_pred.squeeze() - cdr_true).sum().item()
                
                # Classification metrics
                severity_pred_labels = severity_pred.argmax(dim=1)
                correct_predictions += (severity_pred_labels == severity_true).sum().item()
                
                all_severity_true.extend(severity_true.cpu().numpy())
                all_severity_pred.extend(severity_pred_labels.cpu().numpy())
        
        # Calculate metrics
        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)
        avg_cdr_mae = cdr_mae / val_size
        severity_accuracy = correct_predictions / val_size
        
        # Calculate sklearn metrics with zero_division parameter
        f1 = f1_score(all_severity_true, all_severity_pred, average='weighted', zero_division=0)
        precision = precision_score(all_severity_true, all_severity_pred, average='weighted', zero_division=0)
        recall = recall_score(all_severity_true, all_severity_pred, average='weighted', zero_division=0)
        
        # Print epoch results
        print(f"\n=== Epoch {epoch+1}/{epochs} Results ===")
        print(f"Train Loss: {avg_train_loss:.4f}")
        print(f"Val Loss: {avg_val_loss:.4f}")
        print(f"CDR MAE: {avg_cdr_mae:.4f}")
        print(f"Severity Acc: {severity_accuracy:.4f}")
        print(f"F1 Score: {f1:.4f}")
        print(f"Precision: {precision:.4f}")
        print(f"Recall: {recall:.4f}")
        print(f"Learning Rate: {optimizer.param_groups[0]['lr']:.6f}")
        print("="*40)
        
        # Learning rate scheduling
        scheduler.step(avg_val_loss)
        
        # Early stopping
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            early_stopping_counter = 0
            # Save the best model
            torch.save(model.state_dict(), "best_glaucoma_model.pth")
            print("✓ New best model saved!")
        else:
            early_stopping_counter += 1
            print(f"Early stopping counter: {early_stopping_counter}/{patience}")
            if early_stopping_counter >= patience:
                print("Early stopping triggered!")
                break
        
        print()  # Empty line for readability
    
    return model

def main():
    """Main training function"""
    # Set device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    # Set random seeds for reproducibility
    torch.manual_seed(42)
    np.random.seed(42)
    
    # Data path - your specific path
    data_path = "/home/stalin/Projects/ai-eye-disease-detection/dataset/G1020"
    
    # Check if path exists
    if not os.path.exists(data_path):
        print(f"Error: Data path does not exist: {data_path}")
        return
    
    try:
        # Load data
        print("Loading data...")
        img_paths, cdr_values = load_data(data_path)
        
        if len(img_paths) == 0:
            print("No valid images found! Please check your dataset structure.")
            return
        
        # Create data loaders
        print("Creating data loaders...")
        train_loader, val_loader, train_size, val_size = create_data_loaders(
            img_paths, cdr_values, batch_size=16  # Reduced batch size for CPU
        )
        
        print(f"Training samples: {train_size}")
        print(f"Validation samples: {val_size}")
        
        # Initialize model
        print("Initializing model...")
        model = GlaucomaSeverityModel(num_classes=3).to(device)
        
        # Count parameters
        total_params = sum(p.numel() for p in model.parameters())
        trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"Total parameters: {total_params:,}")
        print(f"Trainable parameters: {trainable_params:,}")
        
        # Train model
        print("\nStarting training...")
        print("="*50)
        trained_model = train_model(
            model, train_loader, val_loader, train_size, val_size, 
            epochs=20, device=device
        )
        
        print("\n" + "="*50)
        print("Training completed!")
        print(f"Best model saved as: best_glaucoma_model.pth")
        
    except Exception as e:
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()
        print("Please check your dataset structure and file paths.")

if __name__ == "__main__":
    main()

Using device: cpu
Loading data...
Loading data from: /home/stalin/Projects/ai-eye-disease-detection/dataset/G1020
Metadata file: /home/stalin/Projects/ai-eye-disease-detection/dataset/G1020/OrigaList.csv
Images directory: /home/stalin/Projects/ai-eye-disease-detection/dataset/G1020/Images_Cropped
Metadata shape: (650, 5)
Columns: ['Eye', 'Filename', 'ExpCDR', 'Set', 'Glaucoma']
Found 650 valid images out of 650 total entries
CDR range: 0.161 - 0.963
CDR mean: 0.576 ± 0.116
Severity distribution - Mild: 562, Moderate: 86, Severe: 2
Creating data loaders...
Training samples: 520
Validation samples: 130
Initializing model...
Total parameters: 45,561,412
Trainable parameters: 45,561,412

Starting training...
Epoch 1/20 - Training...
  Batch 10/33, Loss: 0.4856
  Batch 20/33, Loss: 0.3866
  Batch 30/33, Loss: 0.3159
Epoch 1/20 - Validating...

=== Epoch 1/20 Results ===
Train Loss: 0.4106
Val Loss: 0.2931
CDR MAE: 0.4467
Severity Acc: 0.8385
F1 Score: 0.7718
Precision: 0.7150
Recall: 0.8385